# Actividad Práctica 3 — Del EDA al modelo
## Seminario en Ciencia de Datos I (1°C 2026)
**Carrera:** Licenciatura en Ciencia de Datos — Universidad CAECE  
**Docente:** Ing. Fabiana B. Taboada  
**Alumno:** Julian Rachitzky

---

### Caso de estudio
En continuidad con la **Actividad 2**, seguimos trabajando con el **mercado de autos usados**. El objetivo de esta entrega es avanzar desde el EDA hacia la **preparación de datos**, el **modelado básico** y la **publicación** del proyecto.

**Dataset:** *Vehicle Dataset from CardDekho* (Kaggle).  
**Link:** https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho  
Contiene ~301 autos usados con su precio de venta, precio actual de catálogo, año, kilómetros recorridos, tipo de combustible, tipo de vendedor, transmisión y cantidad de dueños previos. Los precios están expresados en **Lakhs de rupias indias (1 Lakh = 100.000 INR)**.

**Problema:** predecir el **precio de venta** de un auto usado (`Selling_Price`) a partir del resto de sus características → es un problema de **regresión**.

> *Ejecución:* el notebook corre de principio a fin. La única condición es que el archivo `car_data.csv` esté disponible (en Colab se sube con el panel de archivos o con `files.upload()`; en Kaggle se agrega el dataset al notebook).


## 0. Importación de librerías


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42   # semilla fija para que los resultados sean reproducibles


## 2.3. Recolección y Preparación de Datos

### Carga del dataset
Cargamos `car_data.csv`. Si el archivo no está en el directorio de trabajo, el bloque lo intenta descargar de un repositorio público para que el notebook pueda ejecutarse de todos modos.


In [ ]:
import os

RUTA = 'car_data.csv'
if os.path.exists(RUTA):
    df = pd.read_csv(RUTA)
else:
    # Fallback: mismo dataset publicado en un repositorio público
    URL = 'https://raw.githubusercontent.com/krishnaik06/simple-Flask-Web-Application/master/car%20data.csv'
    df = pd.read_csv(URL)

print('Dimensiones del dataset:', df.shape)
df.head()


### Revisión inicial de la estructura
Antes de limpiar, revisamos tipos de datos, valores faltantes y duplicados.


In [ ]:
df.info()


In [ ]:
print('Valores nulos por columna:')
print(df.isnull().sum())
print('\nFilas duplicadas:', df.duplicated().sum())


**Lectura:** el dataset no presenta valores nulos y todas las columnas tienen el tipo correcto (numéricas y categóricas). Detectamos **2 filas duplicadas**.


### a) Tratamiento de nulos y duplicados
El dataset no tiene nulos, pero dejamos el tratamiento implementado como buena práctica: las columnas numéricas se rellenarían con la **mediana** y las categóricas con la **moda**. Además eliminamos las filas duplicadas.


In [ ]:
# Relleno preventivo de nulos (no hay, pero queda la lógica documentada)
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Eliminación de duplicados
antes = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print(f'Filas eliminadas por duplicado: {antes - df.shape[0]}')
print('Nulos restantes:', int(df.isnull().sum().sum()))


### b) Tratamiento de un outlier extremo
En la Actividad 2 ya habíamos detectado un registro con un kilometraje desproporcionado. Lo confirmamos y lo quitamos para que no distorsione el modelo.


In [ ]:
print('Km máximos antes del filtro:', df['Kms_Driven'].max())
df = df[df['Kms_Driven'] < 300000].reset_index(drop=True)
print('Km máximos después del filtro:', df['Kms_Driven'].max())
print('Dimensiones finales:', df.shape)


### c) Creación de una variable derivada: `Car_Age`
La columna `Year` por sí sola es poco interpretable. Creamos la **antigüedad del auto** (`Car_Age`), que tiene una relación más directa y lineal con el precio. Usamos como año de referencia **2020** (año de publicación del dataset). *Supuesto declarado: el dataset no informa la fecha exacta de relevamiento; 2020 es el año de publicación en Kaggle.*


In [ ]:
ANIO_REFERENCIA = 2020
df['Car_Age'] = ANIO_REFERENCIA - df['Year']
df[['Year', 'Car_Age', 'Selling_Price']].head()


### d) Estandarización de variables numéricas
Las variables numéricas tienen escalas muy distintas (los kilómetros llegan a cientos de miles, el precio a decenas). Para que sean comparables y los coeficientes del modelo sean interpretables, las **estandarizamos** (media 0, desvío 1) con `StandardScaler`. *El ajuste del scaler se hace más abajo, sólo con datos de entrenamiento, para evitar fuga de información (data leakage).*


## 2.4. Modelado Básico

### Selección de variables (X e y)
- **Variable objetivo (y):** `Selling_Price` (precio de venta, en Lakhs).
- **Variables predictoras (X):** `Present_Price`, `Kms_Driven`, `Car_Age`, `Owner` (numéricas) + `Fuel_Type`, `Seller_Type`, `Transmission` (categóricas).

Las variables categóricas se transforman con **one-hot encoding** (`get_dummies`).


In [ ]:
num_cols = ['Present_Price', 'Kms_Driven', 'Car_Age', 'Owner']
cat_cols = ['Fuel_Type', 'Seller_Type', 'Transmission']

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)
y = df['Selling_Price']

print('Predictoras finales:', list(X.columns))
X.head()


### División en train / test
Reservamos un **20%** de los datos para evaluación, con semilla fija (`random_state=42`).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

print('Train:', X_train.shape[0], 'autos')
print('Test :', X_test.shape[0], 'autos')


### Estandarización (ajustada sólo en train)


In [ ]:
scaler = StandardScaler()
X_train_esc = X_train.copy()
X_test_esc  = X_test.copy()
X_train_esc[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_esc[num_cols]  = scaler.transform(X_test[num_cols])
X_train_esc.head()


### Entrenamiento del modelo: Regresión Lineal


In [ ]:
modelo = LinearRegression()
modelo.fit(X_train_esc, y_train)
y_pred = modelo.predict(X_test_esc)
print('Modelo entrenado correctamente.')


### Evaluación del modelo
Al ser un problema de regresión, usamos **R²** (coeficiente de determinación), **MAE** (error absoluto medio) y **RMSE** (raíz del error cuadrático medio).


In [ ]:
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2_train = r2_score(y_train, modelo.predict(X_train_esc))

print(f'R2  (test)  : {r2:.4f}')
print(f'MAE (test)  : {mae:.4f} Lakhs')
print(f'RMSE (test) : {rmse:.4f} Lakhs')
print(f'R2  (train) : {r2_train:.4f}')


### Visualización 1 — Precio real vs. predicho
Cuanto más cerca de la diagonal roja, mejor la predicción.


In [ ]:
plt.figure(figsize=(6, 4.5))
plt.scatter(y_test, y_pred, alpha=0.7, edgecolor='white', s=55)
lims = [0, max(y_test.max(), y_pred.max()) + 1]
plt.plot(lims, lims, '--', color='red', label='Predicción perfecta')
plt.xlabel('Precio real (Lakhs)')
plt.ylabel('Precio predicho (Lakhs)')
plt.title('Precio real vs. predicho (test)')
plt.legend(); plt.tight_layout(); plt.show()


### Visualización 2 — Coeficientes del modelo
Como las numéricas están estandarizadas, los coeficientes son comparables: indican cuánto sube o baja el precio (en Lakhs) por cada variable.


In [ ]:
coef = pd.Series(modelo.coef_, index=X.columns).sort_values()
plt.figure(figsize=(6, 4.5))
colores = ['red' if v < 0 else 'steelblue' for v in coef.values]
plt.barh(coef.index, coef.values, color=colores)
plt.axvline(0, color='black', lw=0.8)
plt.title('Coeficientes de la regresión lineal')
plt.xlabel('Efecto sobre el precio de venta (Lakhs)')
plt.tight_layout(); plt.show()
coef


### Visualización 3 — Precio según transmisión (relación con la hipótesis de la Actividad 2)


In [ ]:
plt.figure(figsize=(6, 4.5))
sns.boxplot(data=df, x='Transmission', y='Selling_Price',
            hue='Transmission', palette=['steelblue', 'skyblue'], legend=False)
plt.title('Precio de venta según tipo de transmisión')
plt.xlabel('Transmisión'); plt.ylabel('Precio de venta (Lakhs)')
plt.tight_layout(); plt.show()

print(df.groupby('Transmission')['Selling_Price'].mean().round(2))


### Interpretación mínima (obligatoria)

**¿Qué nos dicen las métricas?** El modelo alcanza un **R² de 0.785** en test, es decir explica cerca del **79%** de la variabilidad del precio de venta. El **MAE de 1.38 Lakhs** indica que, en promedio, la predicción se desvía ~1,4 Lakhs del precio real; el **RMSE de 2.41** es mayor que el MAE, lo que delata la existencia de algunos errores grandes en autos de gama alta (la cola de precios altos). El R² de entrenamiento (0.91) es más alto que el de test (0.79): hay un leve sobreajuste, esperable en un modelo lineal con pocos datos (298 autos) y outliers de precio.

**¿Qué limitación veo?** La relación precio–características no es perfectamente lineal y el dataset es chico y desbalanceado (mayoría de autos baratos, manuales y a nafta). Eso hace que el modelo prediga peor los autos caros y poco frecuentes.

**¿Qué haría distinto con más datos/variables?** Probaría modelos no lineales (árboles, Random Forest), incorporaría más variables (marca/modelo, estado, región) y aplicaría transformación logarítmica al precio para amortiguar los valores extremos.

**Relación con la hipótesis de la Actividad 2.** En la Actividad 2 planteamos que *los autos automáticos tienen un precio de venta significativamente mayor que los manuales* (H1) frente a *no hay diferencia* (H0). El modelo lo **apoya**: el coeficiente de `Transmission_Manual` es **negativo (≈ −1.56 Lakhs)** —ser manual baja el precio manteniendo todo lo demás constante— y el precio medio de los automáticos (≈ 9.3 Lakhs) más que duplica al de los manuales (≈ 3.9 Lakhs). La evidencia respalda H1, aunque parte de esa diferencia se explica porque los automáticos suelen ser autos de mayor `Present_Price`.

**Limitaciones éticas / posibles sesgos.** El dataset es de un único mercado (India, precios en Lakhs) y un período acotado, por lo que el modelo **no es trasladable** a otros mercados sin re-entrenamiento. Además, al estar dominado por autos manuales y a nafta, el modelo hereda ese **sesgo de representación** y sería injusto usarlo para tasar segmentos poco presentes (eléctricos, automáticos de gama alta).


## 2.5. Publicación del Código o del Proyecto

Este notebook se publica en un repositorio gratuito para su evaluación. 

> **Enlace de publicación:** https://github.com/julirachi/AP3_RachitzkyJulian_Modelo


## 2.6. Tendencias (resumen)

El eje de la **Unidad 3** que mejor aplica a este caso es **IA / Machine Learning** y, dentro de las herramientas modernas, **MLflow** para la gestión del ciclo de vida del modelo. En esta práctica entrenamos y evaluamos un único modelo, pero en un escenario real probaríamos muchas combinaciones (regresión, árboles, distintos features), y ahí **MLflow Tracking** permite registrar parámetros y métricas (R², MAE, RMSE) de cada corrida y comparar cuál fue la mejor sin perder el rastro.

**Si el caso escalara** —por ejemplo, tasar millones de autos en tiempo real con datos llegando continuamente desde un portal de ventas (más *volumen* y *velocidad*)— el `pandas` + `scikit-learn` de un solo equipo quedaría corto. A nivel conceptual cambiaría a un motor distribuido como **Apache Spark (Spark MLlib)** para entrenar a escala, y usaría **MLflow** para versionar y desplegar el modelo. El desarrollo completo se explica en el informe (punto 2.6).

---
*Fin del notebook — Julian Rachitzky.*
